<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/04_circuit_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

1. Circuit resources versus Trotter timestep

For the same total simulation time, using a smaller timestep means we need more Trotter steps. This usually makes the circuit longer and increases the number of gates, although the exact change depends on how the circuit is built and compiled.

We record the total circuit depth and two-qubit depth to compare the resources needed for different timesteps. This helps us understand the trade-off between getting a more accurate simulation and keeping the circuit practical to run.

Circuit depth alone does not tell us how well a circuit will perform on real hardware, so we examine simulation error and hardware results separately.

In [ ]:
def circuit_depths(qv):

    circuit = qv.qs

    # Total circuit depth
    total_depth = circuit.depth()

    # Two-qubit circuit depth
    def two_qubit_depth_indicator(operation):

        return 1 if operation.num_qubits == 2 else 0

    two_qubit_depth = circuit.depth(
        depth_indicator=two_qubit_depth_indicator
    )

    return total_depth, two_qubit_depth

In [ ]:
qv = build_circuit(
    dt=0.05,
    n_steps=40,
    init_occ=init_occ
)

total_depth, two_qubit_depth = circuit_depths(qv)

print("Total circuit depth:", total_depth)
print("Two-qubit circuit depth:", two_qubit_depth)

In [ ]:
dt_values = [0.05, 0.10, 0.15, 0.20, 0.50]

summary = []

for dt in dt_values:

    # Final simulation time = 3.0
    n_max = int(3.0 / dt)

    # Number of Trotter steps required to reach t = 2
    n_2 = int(2.0 / dt)

    # Circuit at t = 2
    qv_t2 = build_circuit(
        dt,
        n_2,
        init_occ
    )

    total_depth_t2, two_qubit_depth_t2 = circuit_depths(qv_t2)

    # Circuit at final time
    qv_final = build_circuit(
        dt,
        n_max,
        init_occ
    )

    total_depth_final, two_qubit_depth_final = circuit_depths(qv_final)

    summary.append({

        "dt": dt,

        "time_t2": 2.0,
        "total_depth_t2": total_depth_t2,
        "two_qubit_depth_t2": two_qubit_depth_t2,

        "final_time": dt * n_max,
        "total_depth_final": total_depth_final,
        "two_qubit_depth_final": two_qubit_depth_final
    })

depth_df = pd.DataFrame(summary)

print(depth_df)

In [ ]:
T = 1.0
print(f"  {'n_steps':>8}  {'dt':>7}   {'||psi_trotter - psi_exact||':>28}   {'ratio':>6}")
prev = None
psi_ex = la.expm(-1j * H_exact * T) @ psi0
for n_steps in [2, 4, 8, 16, 32]:
    d = T / n_steps
    sv = build_circuit(d, n_steps, init_occ).qs.statevector("array")
    # remove global phase before comparing
    ov = np.vdot(psi_ex, sv)
    err = np.linalg.norm(sv - psi_ex * ov / abs(ov))
    ratio = "" if prev is None else f"{prev/err:5.2f}x"
    print(f"  {n_steps:>8}  {d:>7.4f}   {err:>28.3e}   {ratio:>6}")
    prev = err
print("\nratios near 4 confirm the O(dt^2) global scaling of the mirrored step.")